In [ ]:
#Heavily adapted from https://huggingface.co/amazon/chronos-2 and https://colab.research.google.com/github/amazon-science/chronos-forecasting/blob/main/notebooks/chronos-2-quickstart.ipynb#scrollTo=39de5d7e

In [ ]:
!pip uninstall chronos
!pip uninstall chronos-forecasting 
!pip uninstall chronos-forecast


In [ ]:
pip list

In [2]:
#!pip install "chronos-forecasting[extras]>=2.2" matplotlib
#!pip install "chronos-forecasting>=2.0"
!pip install chronos-forecasting


  Using cached chronos_forecasting-2.3.2-py3-none-any.whl.metadata (24 kB)
Using cached chronos_forecasting-2.3.2-py3-none-any.whl (80 kB)


In [4]:
from chronos import Chronos2Pipeline

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: restrict to one GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Load Chronos‑2
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   # or "cuda" if you want GPU
)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

In [5]:
#import pandas as pd  # requires: pip install 'pandas[pyarrow]'
#from chronos import Chronos2Pipeline


context_df = pd.read_csv("train_df_TTS_Minimal.csv")
future_df = pd.read_csv("test_df_TTS_Minimal.csv")



#remove target var from future:
future_df = future_df.drop(["Target_ConflictFlag"], axis=1)

context_df["id"] = context_df["country"] + "_" + context_df["place"]
future_df["id"]  = future_df["country"] + "_" + future_df["place"]

context_df['periodStart'] = pd.to_datetime(context_df['periodStart'], dayfirst=True)
future_df['periodStart'] = pd.to_datetime(future_df['periodStart'], dayfirst=True)


# Generate predictions with covariates
pred_df = pipeline.predict_df(
    context_df,
    future_df=future_df,
    prediction_length=26,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="Target_ConflictFlag",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)



In [6]:
context_df

,place,country,minBorderDistanceKm,minCapitalDistanceKm,periodStart,LocalEventCount,LocalUniqueActorCount,LocalDistinctEventTypes,LocalDistinctEventSubtypes,LocalTotalFatalities,...,RegionalCiviliansEventCount,RegionalRebelGroupEventCount,RegionalRiotersEventCount,RegionalOtherTypeEventCount,month,season,ConflictFlag,Target_ConflictFlag,DaysSinceMinPeriodStart,id
0,Aba,Central African Republic,61.576638,411.333393,2015-12-31,0,0,0,0,0,...,0,0,0,0,12,winter,0,1,28,Central African Republic_Aba
1,Aba,Central African Republic,61.576638,411.333393,2016-01-28,0,0,0,0,0,...,0,0,0,0,1,winter,0,1,56,Central African Republic_Aba
2,Aba,Central African Republic,61.576638,411.333393,2016-02-25,0,0,0,0,0,...,0,0,0,0,2,winter,0,1,84,Central African Republic_Aba
3,Aba,Central African Republic,61.576638,411.333393,2016-03-24,0,0,0,0,0,...,0,0,0,0,3,spring,0,1,112,Central African Republic_Aba
4,Aba,Central African Republic,61.576638,411.333393,2016-04-21,0,0,0,0,0,...,0,0,0,0,4,spring,1,1,140,Central African Republic_Aba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271,Abba,Central African Republic,60.871993,397.073158,2022-09-01,0,0,0,0,0,...,0,0,0,0,9,autumn,0,1,2464,Central African Republic_Abba
272,Abba,Central African Republic,60.871993,397.073158,2022-09-29,0,0,0,0,0,...,0,1,0,0,9,autumn,1,1,2492,Central African Republic_Abba
273,Abba,Central African Republic,60.871993,397.073158,2022-10-27,2,2,1,1,0,...,0,0,0,0,10,autumn,0,1,2520,Central African Republic_Abba
274,Abba,Central African Republic,60.871993,397.073158,2022-11-24,0,0,0,0,0,...,0,0,0,0,11,autumn,0,1,2548,Central African Republic_Abba


In [ ]:
future_df

In [ ]:
pred_df

In [8]:
#model fine tuning:
from chronos.chronos2 import preprocess

# Prepare data for fine-tuning using the retail sales dataset
train_inputs = preprocess.from_data_frame(
   context_df,
    target_columns=["Target_ConflictFlag"],
    prediction_length=26,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[],
    # Remaining columns are treated as past-only covariates
)


# Fine-tune the model with LoRA
lora_finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=26,
    num_steps=10,#was 1000
    learning_rate=1e-4,
    batch_size=32,
    logging_steps=100,
    finetune_mode="lora",
)




Step,Training Loss


In [10]:
from chronos import Chronos2Pipeline
from chronos.chronos2 import preprocess

# Prepare data for fine-tuning
train_inputs = preprocess.from_data_frame(
    context_df,
    target_columns=["Target_ConflictFlag"],
    prediction_length=26,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[],
)

# LoRA fine-tuning
lora_finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=26,
    num_steps=1000,
    learning_rate=1e-4,
    batch_size=32,
    logging_steps=100,
    finetune_mode="lora",
)




Step,Training Loss


<class 'chronos.chronos2.pipeline.Chronos2Pipeline'>


In [14]:
# ✅ Save the finetuned model
print(type(lora_finetuned_pipeline))
lora_finetuned_pipeline.save_pretrained("chronos2_lora_model")

<class 'chronos.chronos2.pipeline.Chronos2Pipeline'>


In [18]:
#Load lora fnetuned model:
from chronos import Chronos2Pipeline

trained = Chronos2Pipeline.from_pretrained(
    "chronos2_lora_model",
    device_map="cpu"
)

print(type(trained))


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

<class 'chronos.chronos2.pipeline.Chronos2Pipeline'>


In [19]:
#test the loaded model functions properly:

In [21]:
# Generate predictions with covariates
pred_df = trained.predict_df(
    context_df,
    future_df=future_df,
    prediction_length=26,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="Target_ConflictFlag",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df

,id,periodStart,target_name,predictions,0.1,0.5,0.9
0,Central African Republic_Aba,2023-01-19,Target_ConflictFlag,0.997413,0.860482,0.997413,1.007793
1,Central African Republic_Aba,2023-02-16,Target_ConflictFlag,0.997160,0.866492,0.997160,1.009120
2,Central African Republic_Aba,2023-03-16,Target_ConflictFlag,0.996707,0.799804,0.996707,1.009739
3,Central African Republic_Aba,2023-04-13,Target_ConflictFlag,0.997553,0.747868,0.997553,1.009962
4,Central African Republic_Aba,2023-05-11,Target_ConflictFlag,0.998346,0.720419,0.998346,1.009231
...,...,...,...,...,...,...,...
73,Central African Republic_Abba,2024-08-29,Target_ConflictFlag,0.996179,0.017753,0.996179,1.013761
74,Central African Republic_Abba,2024-09-26,Target_ConflictFlag,0.993099,-0.010409,0.993099,1.012529
75,Central African Republic_Abba,2024-10-24,Target_ConflictFlag,0.992473,-0.045462,0.992473,1.014901
76,Central African Republic_Abba,2024-11-21,Target_ConflictFlag,0.991065,-0.085973,0.991065,1.015866
